# OpenAI-compatible RAG test

**QE Perspective:** Validates the full RAG pipeline using the OpenAI SDK against OGX. This tests the same vector store / file_search flow as `test_rag.ipynb` but via the OpenAI-compatible API, verifying that third-party clients can use RAG features.

- **Embeddings:** Create embeddings via OpenAI client, verify correct dimension.
- **Vector store lifecycle:** Create, upload file, attach with chunking, search, delete.
- **RAG query:** Responses API with file_search tool, verify answer is grounded.
- **Comparison:** Same question with vs without vector store.

Config: `BASE_URL`, `MODEL`, `EMBEDDING_MODEL`, `EMBEDDING_DIMENSION`.

## Setup

In [ ]:
import os
from scripts.helpers import response_text

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
openai_base_url = base_url.rstrip("/") + "/v1"
model = os.environ.get("INFERENCE_MODEL")
embedding_model = os.environ.get("EMBEDDING_MODEL")
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION", "768"))

_skip_reason = None

assert base_url, "BASE_URL must be set"
assert model, "MODEL must be set"
if not embedding_model:
    _skip_reason = (
        "EMBEDDING_MODEL not set \u2014 OpenAI RAG test requires an embedding model"
    )
    print(f"SKIPPED: {_skip_reason}")

In [ ]:
if not _skip_reason:
    from openai import OpenAI

    client = OpenAI(api_key="no-key-needed", base_url=openai_base_url)

    providers_resp = client.get("/providers", cast_to=object)
    vector_providers = [
        p for p in providers_resp.get("data", []) if p.get("api") == "vector_io"
    ]
    if not vector_providers:
        _skip_reason = "No vector_io provider available"
        print(f"SKIPPED: {_skip_reason}")
    else:
        vector_provider_id = vector_providers[0]["provider_id"]
        print(f"Using vector_io provider: {vector_provider_id}")

## Embeddings API

Verify the embedding model produces vectors of the expected dimension.

In [ ]:
if not _skip_reason:
    emb_response = client.embeddings.create(
        input="Test embedding for RAG pipeline",
        model=embedding_model,
    )
    assert len(emb_response.data) > 0, "Embeddings response has no data"
    embedding = emb_response.data[0].embedding
    assert isinstance(embedding, list), "Embedding should be a list of floats"
    assert len(embedding) == embedding_dimension, (
        f"Expected dimension {embedding_dimension}, got {len(embedding)}"
    )
    print(f"Embedding dimension: {len(embedding)} (expected {embedding_dimension})")

## Vector store + file upload + RAG query

Create a vector store, upload a document, attach it, then query via Responses API with file_search.

In [ ]:
if not _skip_reason:
    from io import BytesIO
    from uuid import uuid4

    doc_text = (
        "Acme Corp Annual Report 2024. "
        "Total revenue reached $4.2 billion, a 15% increase year-over-year. "
        "The Cloud Services division generated $1.8 billion, growing 23%. "
        "Free cash flow was $890 million. "
        "The company announced acquisition of DataFlow Inc. for $350 million "
        "on September 15, 2024."
    )

    vector_store = None
    uploaded_file = None

    vector_store = client.vector_stores.create(
        name=f"openai_rag_test_{uuid4().hex[:8]}",
        extra_body={
            "embedding_model": embedding_model,
            "provider_id": vector_provider_id,
        },
    )
    assert vector_store.id, "Vector store creation failed"
    print(f"Created vector store: {vector_store.id}")

    file_buffer = BytesIO(doc_text.encode("utf-8"))
    file_buffer.name = "acme_report.txt"
    uploaded_file = client.files.create(file=file_buffer, purpose="assistants")
    assert uploaded_file.id, "File upload failed"
    print(f"Uploaded file: {uploaded_file.id}")

    vs_file = client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=uploaded_file.id,
        chunking_strategy={
            "type": "static",
            "static": {"max_chunk_size_tokens": 256, "chunk_overlap_tokens": 32},
        },
    )
    print(f"File attached, status: {vs_file.status}")

## Vector store search

Search the vector store directly to verify document was indexed.

In [ ]:
if not _skip_reason and vector_store:
    search_resp = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query="Acme Corp revenue",
        max_num_results=3,
    )
    assert hasattr(search_resp, "data"), "Search response missing data"
    assert len(search_resp.data) > 0, "Search returned no results"
    print(f"Search returned {len(search_resp.data)} result(s)")

## RAG query via Responses API

Query using file_search and assert the answer contains facts from the document.

In [ ]:
if not _skip_reason and vector_store:
    rag_response = client.responses.create(
        model=model,
        instructions="Answer using only the provided documents. Be concise.",
        input=[
            {"role": "user", "content": "What was Acme Corp's total revenue in 2024?"}
        ],
        tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
        tool_choice={"type": "file_search"},
        stream=False,
    )
    assert rag_response.status == "completed", (
        f"Expected completed, got {rag_response.status}"
    )
    rag_text = response_text(rag_response)
    assert rag_text, "Expected non-empty RAG response"
    assert "4.2" in rag_text or "4,2" in rag_text or "billion" in rag_text.lower(), (
        f"Expected revenue figure in answer, got: {rag_text[:300]}"
    )
    print(f"RAG answer: {rag_text[:200]}")

## Comparison: with vs without vector store

The model should not know about Acme Corp without the vector store context.

In [ ]:
if not _skip_reason and vector_store:
    q = "Who did Acme Corp acquire on September 15, 2024?"

    resp_without = client.responses.create(
        model=model,
        input=q,
        stream=False,
    )
    text_without = response_text(resp_without) or ""

    resp_with = client.responses.create(
        model=model,
        instructions="Answer strictly from the provided documents.",
        input=[{"role": "user", "content": q}],
        tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
        tool_choice={"type": "file_search"},
        stream=False,
    )
    text_with = response_text(resp_with) or ""

    assert resp_with.status == "completed"
    assert "DataFlow" in text_with or "350" in text_with, (
        f"Expected acquisition details in RAG answer, got: {text_with[:300]}"
    )
    print(f"Without RAG: {text_without[:150]}")
    print(f"With RAG:    {text_with[:150]}")

## Cleanup

In [ ]:
if not _skip_reason:
    if vector_store:
        try:
            client.vector_stores.delete(vector_store_id=vector_store.id)
            print(f"Deleted vector store: {vector_store.id}")
        except Exception as e:
            print(f"Warning: {e}")
    if uploaded_file:
        try:
            client.files.delete(file_id=uploaded_file.id)
            print(f"Deleted file: {uploaded_file.id}")
        except Exception as e:
            print(f"Warning: {e}")